# XRD/LRD Cosmology Toolkit

X-RAY DOT / LITTLE RED DOT COSMOLOGICAL EVOLUTION TOOLKIT — v1.0
A comprehensive Python module for:
  1. LRD → XRD → AGN evolutionary sequence modeling
  2. SMBH seed growth and gas cocoon clearing physics
  3. TNG-style halo assembly timeline comparison
  4. Assembly Index (A_c) for early-universe substructure
  5. SED fitting and obscuration modeling
  6. Redshift evolution and clustering analysis

Optimized for Google Colab / Jupyter Notebook usage.
Tested 2026-05-09 with 1000 synthetic objects.

Author: Cloud-9 Assembly Project

## Colab quick start

Run the setup cell first, then execute the remaining cells top to bottom.


In [ ]:
# Optional: install/upgrade dependencies in Colab if needed
# Uncomment the next line if any package is missing
# !pip install numpy pandas matplotlib seaborn scipy scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity, NearestNeighbors
from sklearn.cross_decomposition import CCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


## COSMOLOGY


In [ ]:
class Cosmology:
    def __init__(self, H0=67.4, Omega_m=0.315, Omega_L=0.685):
        self.H0 = H0
        self.Omega_m = Omega_m
        self.Omega_L = Omega_L
        self.h = H0 / 100.0
        self.age_Gyr = 13.8

    def age_at_z(self, z):
        return self.age_Gyr / (1 + z)**1.5 * (self.Omega_m**0.5 + self.Omega_L**0.5 * (1+z)**(-1.5))**(-1)

    def lookback_time(self, z):
        return self.age_Gyr - self.age_at_z(z)

    def luminosity_distance(self, z):
        c = 299792.458
        return c * z / self.H0 * (1 + z/2) * (1 + z)

    def comoving_distance(self, z):
        return self.luminosity_distance(z) / (1 + z)


## LRD / XRD / AGN EVOLUTIONARY MODEL


In [ ]:
class LRDEvolutionModel:
    def __init__(self, cosmology=None):
        self.cosmo = cosmology if cosmology else Cosmology()
        self.M_BH_seed = 1e4
        self.M_BH_final = 1e9
        self.M_stellar_host = 1e7
        self.r_e = 250
        self.T_eff = 6400
        self.L_X_threshold = 1e43
        self.tau_cocoon_initial = 100
        self.tau_cocoon_final = 0.1
        self.cocoon_clearing_timescale = 0.3

    def gas_cocoon_optical_depth(self, t, t_form):
        dt = t - t_form
        if dt < 0:
            return self.tau_cocoon_initial
        tau = self.tau_cocoon_initial * np.exp(-dt / self.cocoon_clearing_timescale)
        return max(tau, self.tau_cocoon_final)

    def xray_transmission(self, tau):
        f_patchy = min(0.5, 0.1 + 0.4 * (1 - tau / self.tau_cocoon_initial))
        tau_eff = tau * (1 - f_patchy) + 0.5 * f_patchy
        return np.exp(-tau_eff)

    def classify_phase(self, z, M_BH, tau):
        L_X = self.estimate_xray_luminosity(M_BH)
        f_trans = self.xray_transmission(tau)
        L_X_obs = L_X * f_trans
        if L_X_obs < 1e42:
            return 'LRD'
        elif L_X_obs < self.L_X_threshold:
            return 'XRD'
        else:
            return 'AGN'

    def estimate_xray_luminosity(self, M_BH, accretion_rate=0.3):
        L_Edd = 1.26e38 * M_BH
        L_bol = accretion_rate * L_Edd
        L_X = 0.1 * L_bol
        return np.clip(L_X, 1e30, 1e48)

    def sed_model(self, wavelengths, z, M_BH, tau, T_gas=6400):
        h = 6.626e-27
        c = 2.998e10
        k_B = 1.381e-16
        nu = c / (wavelengths * 1e-8)
        B_nu = (2 * h * nu**3 / c**2) / (np.exp(h * nu / (k_B * T_gas)) - 1)
        f_AGN = self.xray_transmission(tau)
        alpha = -1.5
        L_AGN = self.estimate_xray_luminosity(M_BH)
        F_AGN = L_AGN * (nu / 1e18)**alpha
        flux = B_nu * (1 - f_AGN) + F_AGN * f_AGN
        D_L = self.cosmo.luminosity_distance(z) * 3.086e24
        flux = flux / (4 * np.pi * D_L**2) * (1 + z)
        return flux

    def evolve_population(self, z_range=(6, 2), n_objects=1000, random_state=42):
        rng = np.random.RandomState(random_state)
        z_form = rng.uniform(z_range[1], z_range[0], n_objects)
        t_form = np.array([self.cosmo.age_at_z(z) for z in z_form])
        dt = rng.exponential(0.8, n_objects)
        t_now = t_form + dt
        z_now = []
        for t in t_now:
            z_inv = max(0, (self.cosmo.age_Gyr / t)**(2/3) - 1)
            z_now.append(max(z_range[1], min(z_range[0], z_inv)))
        z_now = np.array(z_now)
        log_M_BH = rng.uniform(4, 9, n_objects)
        M_BH = 10**log_M_BH
        tau = [self.gas_cocoon_optical_depth(t, tf) for t, tf in zip(t_now, t_form)]
        phases = [self.classify_phase(z, m, t) for z, m, t in zip(z_now, M_BH, tau)]
        L_X = [self.estimate_xray_luminosity(m) * self.xray_transmission(t) for m, t in zip(M_BH, tau)]

        df = pd.DataFrame({
            'z_form': z_form,
            'z_now': z_now,
            't_form_Gyr': t_form,
            't_now_Gyr': t_now,
            'M_BH': M_BH,
            'tau_cocoon': tau,
            'phase': phases,
            'L_X': L_X,
            'log_L_X': np.log10(np.clip(L_X, 1e30, 1e48)),
            'f_AGN': [self.xray_transmission(t) for t in tau],
            'M_stellar': rng.lognormal(np.log(self.M_stellar_host), 0.5, n_objects)
        })
        return df


## TNG HALO ASSEMBLY COMPARISON


In [ ]:
class TNGAssemblyComparator:
    def __init__(self, cosmology=None):
        self.cosmo = cosmology if cosmology else Cosmology()
        self.box_size = 110.7
        self.mass_resolution = 1.4e6
        self.snapshot_z99 = 0.0

    def halo_mass_at_z(self, M_z0, z):
        return M_z0 * (1 + z)**(-0.5) * np.exp(-z / 3.0)

    def bh_halo_mass_relation(self, M_halo, z, alpha=1.5):
        beta = 0.5
        M_BH = 1e-3 * M_halo**alpha / (1 + z)**beta
        return np.clip(M_BH, 1e4, 1e10)

    def merger_rate(self, z, M_halo):
        return 0.01 * (M_halo / 1e12)**0.3 * (1 + z)**2.5

    def assembly_time(self, M_halo, z):
        z_form = 2.0 * (M_halo / 1e12)**(-0.1) * (1 + z) - 1
        return self.cosmo.age_at_z(z_form)

    def compare_with_lrd(self, lrd_df, M_halo_z0=1e12):
        results = {}
        M_BH_tng = [self.bh_halo_mass_relation(self.halo_mass_at_z(M_halo_z0, z), z) for z in lrd_df['z_now']]
        ratio = lrd_df['M_BH'].values / np.array(M_BH_tng)
        results['M_BH_ratio_mean'] = np.mean(ratio)
        results['M_BH_ratio_std'] = np.std(ratio)
        merger_rates = [self.merger_rate(z, self.halo_mass_at_z(M_halo_z0, z)) for z in lrd_df['z_now']]
        results['merger_rate_mean'] = np.mean(merger_rates)
        t_ass = [self.assembly_time(self.halo_mass_at_z(M_halo_z0, z), z) for z in lrd_df['z_now']]
        results['assembly_time_mean_Gyr'] = np.mean(t_ass)
        return results


## ASSEMBLY INDEX (A_c)


In [ ]:
class AssemblyIndexCalculator:
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def chemical_entropy(self, df, feature_cols=None):
        if feature_cols is None:
            feature_cols = ['z_now', 'M_BH', 'tau_cocoon', 'L_X']
        X = df[feature_cols].values
        X = StandardScaler().fit_transform(X)
        kde = KernelDensity(bandwidth=0.5, kernel='gaussian')
        kde.fit(X)
        log_dens = kde.score_samples(X)
        return -np.mean(log_dens)

    def kinematic_complexity(self, df):
        vels = df[['z_now', 'log_L_X']].values
        vels = vels + np.random.normal(0, 1e-6, vels.shape)
        cov = np.cov(vels.T)
        eigenvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
        if eigenvals[0] > 1e-10:
            anisotropy = 1 - eigenvals[1] / eigenvals[0]
        else:
            anisotropy = 0
        complexity = np.sqrt(np.sum(np.maximum(eigenvals, 0))) * (1 + abs(anisotropy))
        return complexity, anisotropy

    def topological_complexity(self, df, feature_cols=None):
        if feature_cols is None:
            feature_cols = ['z_now', 'M_BH', 'tau_cocoon']
        X = df[feature_cols].values
        X = StandardScaler().fit_transform(X)
        if len(X) < self.n_neighbors + 1:
            return 0.0
        nbrs = NearestNeighbors(n_neighbors=min(self.n_neighbors+1, len(X))).fit(X)
        distances, _ = nbrs.kneighbors(X)
        nn_distances = distances[:, 1:].mean(axis=1)
        return abs(stats.skew(np.log(nn_distances + 1e-10)))

    def integrated_information(self, df, feature_cols=None):
        if feature_cols is None:
            feature_cols = ['z_now', 'M_BH', 'tau_cocoon']
        X_phys = df[feature_cols].values
        X_obs = df[['L_X', 'M_stellar']].values
        X_phys = StandardScaler().fit_transform(X_phys)
        X_obs = StandardScaler().fit_transform(X_obs)
        n_comp = min(2, X_phys.shape[1], X_obs.shape[1])
        cca = CCA(n_components=n_comp)
        cca.fit(X_phys, X_obs)
        X_c, X_k = cca.transform(X_phys, X_obs)
        mi_proxy = sum(np.corrcoef(X_c[:, i], X_k[:, i])[0, 1]**2 for i in range(n_comp))
        return mi_proxy / n_comp

    def compute_ac(self, df, feature_cols=None, weights=None):
        if weights is None:
            weights = {'chemical': 0.3, 'kinematic': 0.25, 'topological': 0.25, 'information': 0.2}
        S_chem = self.chemical_entropy(df, feature_cols)
        K_comp, aniso = self.kinematic_complexity(df)
        T_comp = self.topological_complexity(df, feature_cols)
        I_info = self.integrated_information(df, feature_cols)
        A_c = (
            weights['chemical'] * (1.0 / (1.0 + S_chem)) +
            weights['kinematic'] * np.tanh(K_comp / 100) +
            weights['topological'] * np.tanh(T_comp / 2) +
            weights['information'] * I_info
        )
        return {
            'A_c': A_c, 'chemical_entropy': S_chem, 'kinematic_complexity': K_comp,
            'anisotropy': aniso, 'topological_complexity': T_comp,
            'integrated_information': I_info, 'weights': weights, 'n_objects': len(df)
        }


## VISUALIZATION


In [ ]:
def plot_evolution_sequence(model, save_path=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    z_range = np.linspace(6, 2, 100)
    t_range = [model.cosmo.age_at_z(z) for z in z_range]
    M_BH = model.M_BH_seed * np.exp(np.linspace(0, 3, 100))
    tau = [model.gas_cocoon_optical_depth(t, t_range[0]) for t in t_range]
    phases = [model.classify_phase(z, m, t) for z, m, t in zip(z_range, M_BH, tau)]
    L_X = [model.estimate_xray_luminosity(m) * model.xray_transmission(t) for m, t in zip(M_BH, tau)]

    ax = axes[0, 0]
    ax.semilogy(z_range, M_BH, 'k-', linewidth=2)
    ax.set_xlabel('Redshift z')
    ax.set_ylabel(r'$M_{BH}$ [$M_\odot$]')
    ax.set_title('BH Mass Growth')
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.semilogy(z_range, tau, 'b-', linewidth=2)
    ax.axhline(1, color='r', linestyle='--', alpha=0.5, label='Unity optical depth')
    ax.set_xlabel('Redshift z')
    ax.set_ylabel(r'$	au_{cocoon}$')
    ax.set_title('Gas Cocoon Clearing')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.semilogy(z_range, L_X, 'g-', linewidth=2)
    ax.axhline(model.L_X_threshold, color='r', linestyle='--', alpha=0.5, label='XRD threshold')
    ax.set_xlabel('Redshift z')
    ax.set_ylabel(r'$L_X$ [erg/s]')
    ax.set_title('X-ray Luminosity Evolution')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    phase_colors = {'LRD': 'red', 'XRD': 'orange', 'AGN': 'blue'}
    for i, (z, phase) in enumerate(zip(z_range, phases)):
        ax.scatter(z, i, c=phase_colors[phase], s=50, alpha=0.8)
    ax.set_xlabel('Redshift z')
    ax.set_ylabel('Evolutionary time')
    ax.set_title('Phase Evolution')
    ax.set_yticks([])
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=p) for p, c in phase_colors.items()]
    ax.legend(handles=legend_elements, loc='upper left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_population_properties(df, save_path=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    for phase, group in df.groupby('phase'):
        ax.hist(group['z_now'], bins=20, alpha=0.6, label=phase, edgecolor='k')
    ax.set_xlabel('Redshift z')
    ax.set_ylabel('Count')
    ax.set_title('Redshift Distribution by Phase')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    for phase, group in df.groupby('phase'):
        ax.scatter(group['z_now'], group['log_L_X'], label=phase, s=30, alpha=0.6, edgecolors='k')
    ax.set_xlabel('Redshift z')
    ax.set_ylabel(r'$\log L_X$ [erg/s]')
    ax.set_title('X-ray Luminosity vs Redshift')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    for phase, group in df.groupby('phase'):
        ax.scatter(group['M_stellar'], group['M_BH'], label=phase, s=30, alpha=0.6, edgecolors='k')
    ax.set_xlabel(r'$M_*$ [$M_\odot$]')
    ax.set_ylabel(r'$M_{BH}$ [$M_\odot$]')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title('BH Mass vs Stellar Mass')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.hist(df['tau_cocoon'], bins=30, color='steelblue', edgecolor='k', alpha=0.7)
    ax.set_xlabel(r'$	au_{cocoon}$')
    ax.set_ylabel('Count')
    ax.set_title('Gas Cocoon Optical Depth Distribution')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_ac_comparison(ac_results, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    metrics = ['A_c', 'chemical_entropy', 'kinematic_complexity', 'anisotropy', 'topological_complexity', 'integrated_information']
    labels = ['A_c', 'S_chem', 'K_comp', 'Anisotropy', 'T_comp', 'I_info']
    values = [ac_results[m] for m in metrics]
    colors = ['steelblue' if v > np.median(values) else 'coral' for v in values]
    bars = ax.bar(labels, values, color=colors, edgecolor='k', alpha=0.8)
    ax.set_ylabel('Value')
    ax.set_title(f'Assembly Index Components (A_c = {ac_results["A_c"]:.3f})')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


## FULL PIPELINE


In [ ]:
def run_xrd_pipeline(n_objects=1000, random_state=42):
    print("="*60)
    print("X-RAY DOT / LRD COSMOLOGICAL EVOLUTION PIPELINE")
    print("="*60)

    cosmo = Cosmology()
    lrd_model = LRDEvolutionModel(cosmology=cosmo)
    tng_comp = TNGAssemblyComparator(cosmology=cosmo)
    ac_calc = AssemblyIndexCalculator()

    print("\n[1/5] Generating synthetic LRD/XRD/AGN population...")
    population = lrd_model.evolve_population(z_range=(6, 2), n_objects=n_objects, random_state=random_state)
    print(f"  Generated {len(population)} objects")
    print(f"  LRDs: {sum(population['phase'] == 'LRD')}")
    print(f"  XRDs: {sum(population['phase'] == 'XRD')}")
    print(f"  AGNs: {sum(population['phase'] == 'AGN')}")

    print("\n[2/5] Computing Assembly Index...")
    ac_results = ac_calc.compute_ac(population)
    print(f"  A_c = {ac_results['A_c']:.3f}")
    print(f"  Chemical entropy: {ac_results['chemical_entropy']:.3f}")
    print(f"  Kinematic complexity: {ac_results['kinematic_complexity']:.1f}")
    print(f"  Anisotropy: {ac_results['anisotropy']:.3f}")
    print(f"  Topological complexity: {ac_results['topological_complexity']:.3f}")
    print(f"  Integrated information: {ac_results['integrated_information']:.3f}")

    print("\n[3/5] Comparing with TNG halo assembly...")
    tng_results = tng_comp.compare_with_lrd(population, M_halo_z0=1e12)
    print(f"  M_BH ratio (model/TNG): {tng_results['M_BH_ratio_mean']:.2e} ± {tng_results['M_BH_ratio_std']:.2e}")
    print(f"  Mean merger rate: {tng_results['merger_rate_mean']:.3f} Gyr^-1")
    print(f"  Mean assembly time: {tng_results['assembly_time_mean_Gyr']:.2f} Gyr")

    print("\n[4/5] Generating plots...")
    plot_evolution_sequence(lrd_model, save_path='/mnt/agents/output/xrd_evolution_sequence.png')
    plot_population_properties(population, save_path='/mnt/agents/output/xrd_population.png')
    plot_ac_comparison(ac_results, save_path='/mnt/agents/output/xrd_ac_components.png')

    print("\n[5/5] Computing phase statistics...")
    phase_stats = population.groupby('phase').agg({
        'z_now': ['mean', 'std', 'min', 'max'],
        'M_BH': ['mean', 'std'],
        'L_X': ['mean', 'std'],
        'tau_cocoon': ['mean', 'std']
    })
    print(phase_stats)

    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)

    return {
        'population': population,
        'ac_results': ac_results,
        'tng_comparison': tng_results,
        'phase_statistics': phase_stats
    }


## Example run

If you want to execute the full pipeline interactively in Colab, run the cell below after all definitions are loaded.


In [ ]:
# Example execution
results = run_xrd_pipeline(n_objects=1000, random_state=42)
results.keys()
